In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT_PATH = Path.cwd().parent

if str(PROJECT_ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_PATH))

from configs.config import LANDING_PATH

sample_transactions_file = (
    LANDING_PATH / "User0_credit_card_transactions.csv"
)

print("Sample file:", sample_transactions_file)
print("File exists:", sample_transactions_file.exists())
import pandas as pd

transactions_sample_df = pd.read_csv(sample_transactions_file)

transactions_sample_df.head()
print("Rows and columns:", transactions_sample_df.shape)
print("\nColumn names:")
print(transactions_sample_df.columns.tolist())
transactions_sample_df.info()
transactions_sample_df.isna().sum()
transactions_sample_df.nunique()
transactions_sample_df.describe(include="all").T
missing_values_df = (
    transactions_sample_df
    .isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_values_df["missing_percentage"] = (
    missing_values_df["missing_count"]
    / len(transactions_sample_df)
    * 100
).round(2)

missing_values_df.sort_values(
    by="missing_percentage",
    ascending=False
)
categorical_columns = [
    "Use Chip",
    "Merchant State",
    "Errors?",
    "Is Fraud?",
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(
        transactions_sample_df[column]
        .value_counts(dropna=False)
        .head(20)
    )
    duplicate_row_count = transactions_sample_df.duplicated().sum()

print("Exact duplicate rows:", duplicate_row_count)

from configs.source_schemas import TRANSACTION_REQUIRED_COLUMNS
from src.schema_validation import validate_columns

schema_result = validate_columns(
    actual_columns=transactions_sample_df.columns.tolist(),
    expected_columns=TRANSACTION_REQUIRED_COLUMNS,
)

print("Expected column count:", schema_result["expected_count"])
print("Actual column count:", schema_result["actual_count"])
print("Missing columns:", schema_result["missing_columns"])
print("Unexpected columns:", schema_result["unexpected_columns"])
print("Schema valid:", schema_result["is_valid"])

from configs.source_schemas import TRANSACTION_EXPECTED_DTYPES
from src.schema_validation import validate_dtypes

actual_dtypes = {
    column: str(dtype)
    for column, dtype in transactions_sample_df.dtypes.items()
}

dtype_result = validate_dtypes(
    actual_dtypes=actual_dtypes,
    expected_dtypes=TRANSACTION_EXPECTED_DTYPES,
)

print("Data types valid:", dtype_result["is_valid"])
print("Mismatched columns:", dtype_result["mismatched_columns"])

